In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os, json, shutil, importlib
sys.path.append("/content/drive/MyDrive/tcc")
from src import config, modelos, treino
importlib.reload(modelos); importlib.reload(treino)

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NENHUMA — troque o ambiente para T4")

Mounted at /content/drive
GPU: Tesla T4


In [ ]:
ORIGEM = f"{config.ELA_DIR}/v2"
APARELHOS = ["samsung_a56", "samsung_s23fe", "canon_powershot_sx510hs"]
HIPER = {"lr": 1e-4, "epocas": 20, "paciencia": 4, "tamanho_lote": 64}
N = config.CROPS_TREINO
os.makedirs("/content/dados_v2", exist_ok=True)


def carregar(nome, entrada):
    local = f"/content/dados_v2/{nome}_{entrada}.npy"
    if not os.path.exists(local):
        shutil.copy(f"{ORIGEM}/{nome}_{entrada}.npy", local)
    return np.load(local, mmap_mode="r"), pd.read_csv(f"{ORIGEM}/{nome}_rotulos.csv")


def montar_rodada(aparelho_teste, entrada):
    Xs_tr, ys_tr, Xs_val, ys_val = [], [], [], []

    X, rot = carregar("treino_base", entrada)
    Xs_tr.append(np.asarray(X)); ys_tr.append(rot["rotulo"].values)
    X, rot = carregar("val_base", entrada)
    Xs_val.append(np.asarray(X)); ys_val.append(rot["rotulo"].values)

    crops_reais_novos = 0
    for ap in APARELHOS:
        if ap == aparelho_teste:
            continue
        X, rot = carregar(f"disp_aug_{ap}", entrada)
        tr = (rot["particao_foto"] == "treino").values
        Xs_tr.append(X[tr]);  ys_tr.append(rot["rotulo"].values[tr])
        Xs_val.append(X[~tr]); ys_val.append(rot["rotulo"].values[~tr])
        crops_reais_novos += int(tr.sum())

    # IA extra na mesma quantidade das fotos novas (sorteio fixo por rodada)
    X, rot = carregar("ia_extra", entrada)
    rng = np.random.default_rng(config.SEED + APARELHOS.index(aparelho_teste))
    escolhidos = rng.choice(rot["id"].unique(), crops_reais_novos // N, replace=False)
    m = rot["id"].isin(escolhidos).values
    Xs_tr.append(X[m]); ys_tr.append(rot["rotulo"].values[m])

    X_tr = torch.from_numpy(np.concatenate(Xs_tr)); y_tr = torch.from_numpy(np.concatenate(ys_tr)).long()
    X_val = torch.from_numpy(np.concatenate(Xs_val)); y_val = torch.from_numpy(np.concatenate(ys_val)).long()
    return X_tr, y_tr, X_val, y_val

In [ ]:
X_tr, y_tr, X_val, y_val = montar_rodada("samsung_a56", "rgb")
print("Treino:", tuple(X_tr.shape), "| reais:", int((y_tr == 0).sum()), "| IA:", int(y_tr.sum()))
print("Val:   ", tuple(X_val.shape), "| reais:", int((y_val == 0).sum()), "| IA:", int(y_val.sum()))
del X_tr, y_tr, X_val, y_val

Treino: (6336, 256, 256, 3) | reais: 3168 | IA: 3168
Val:    (1268, 256, 256, 3) | reais: 668 | IA: 600


In [4]:
CONFIGS = [
    (entrada, aparelho, seed)
    for seed in [42, 7, 123]
    for aparelho in APARELHOS
    for entrada in ["rgb", "ela"]
]

for i, (entrada, aparelho, seed) in enumerate(CONFIGS, 1):
    NOME = f"resnet18_{entrada}_v2_sem_{aparelho}_seed{seed}"
    PASTA = f"{config.MODELOS}/{NOME}"
    print(f"\n===== [{i}/{len(CONFIGS)}] {NOME} =====")

    X_tr, y_tr, X_val, y_val = montar_rodada(aparelho, entrada)
    print(f"Treino {tuple(X_tr.shape)} | Val {tuple(X_val.shape)}")

    treino.fixar_seed(seed)
    modelo = modelos.criar_modelo("resnet18")
    os.makedirs(PASTA, exist_ok=True)
    json.dump({"modelo": "resnet18", "entrada": entrada, "experimento": f"v2_sem_{aparelho}",
               "aparelho_teste": aparelho, "seed": seed, **HIPER},
              open(f"{PASTA}/config.json", "w"), indent=2)

    hist = treino.treinar(modelo, X_tr, y_tr, X_val, y_val, PASTA, seed=seed, **HIPER)

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    for col, eixo, rotulo in [("perda", a1, "Perda"), ("acc", a2, "Acurácia")]:
        eixo.plot(hist["epoca"], hist[f"{col}_treino"], label="treino")
        eixo.plot(hist["epoca"], hist[f"{col}_val"], label="validação")
        eixo.set_xlabel("Época"); eixo.set_ylabel(rotulo); eixo.legend()
    a1.set_title(NOME)
    plt.tight_layout(); plt.savefig(f"{PASTA}/curvas.png", dpi=150); plt.close(fig)

    del modelo, X_tr, y_tr, X_val, y_val
    torch.cuda.empty_cache()

print("\nTodas as rodadas concluídas.")


===== [1/18] resnet18_rgb_v2_sem_samsung_a56_seed42 =====
Treino (6336, 256, 256, 3) | Val (1268, 256, 256, 3)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 200MB/s]


Época  1 | treino: perda 0.1357 acc 0.945 | val: perda 0.1399 acc 0.946 | 12.3s  <- melhor
Época  2 | treino: perda 0.0553 acc 0.980 | val: perda 0.1253 acc 0.957 | 10.2s  <- melhor
Época  3 | treino: perda 0.0407 acc 0.985 | val: perda 0.1560 acc 0.954 | 10.9s
Época  4 | treino: perda 0.0299 acc 0.989 | val: perda 0.2162 acc 0.950 | 10.7s
Época  5 | treino: perda 0.0219 acc 0.993 | val: perda 0.1581 acc 0.956 | 10.8s
Época  6 | treino: perda 0.0239 acc 0.991 | val: perda 0.1432 acc 0.960 | 11.2s
Early stopping: 4 épocas sem melhora na validação.

===== [2/18] resnet18_ela_v2_sem_samsung_a56_seed42 =====
Treino (6336, 256, 256, 3) | Val (1268, 256, 256, 3)
Época  1 | treino: perda 0.3579 acc 0.840 | val: perda 0.3264 acc 0.845 | 11.3s  <- melhor
Época  2 | treino: perda 0.2491 acc 0.898 | val: perda 0.3623 acc 0.843 | 11.9s
Época  3 | treino: perda 0.2110 acc 0.917 | val: perda 0.3403 acc 0.869 | 12.4s
Época  4 | treino: perda 0.1711 acc 0.934 | val: perda 0.2973 acc 0.881 | 12.2s  <- 